In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 12


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.868295282125473
Epoch 2/100, Loss: 2.810212306678295
Epoch 3/100, Loss: 2.549817107617855
Epoch 4/100, Loss: 2.8106565177440643
Epoch 5/100, Loss: 2.7606869488954544
Epoch 6/100, Loss: 2.8119011372327805
Epoch 7/100, Loss: 2.6536411494016647
Epoch 8/100, Loss: 2.844179183244705
Epoch 9/100, Loss: 2.7065969109535217
Epoch 10/100, Loss: 2.7378532215952873
Epoch 11/100, Loss: 2.7636166140437126
Epoch 12/100, Loss: 2.9095917493104935
Epoch 13/100, Loss: 2.7244213297963142
Epoch 14/100, Loss: 2.880185216665268
Epoch 15/100, Loss: 2.5719250068068504


Epoch 16/100, Loss: 3.000906892120838
Epoch 17/100, Loss: 2.828748159110546
Epoch 18/100, Loss: 2.695861093699932
Epoch 19/100, Loss: 2.735436387360096
Epoch 20/100, Loss: 2.537631541490555
Epoch 21/100, Loss: 2.720295250415802
Epoch 22/100, Loss: 2.7159406542778015
Epoch 23/100, Loss: 2.7967401295900345
Epoch 24/100, Loss: 2.760750725865364
Epoch 25/100, Loss: 2.6472487673163414
Epoch 26/100, Loss: 2.671936795115471
Epoch 27/100, Loss: 2.6805568486452103
Epoch 28/100, Loss: 2.832126021385193
Epoch 29/100, Loss: 2.765418477356434
Epoch 30/100, Loss: 2.683061070740223


Epoch 31/100, Loss: 2.770990289747715
Epoch 32/100, Loss: 2.790893390774727
Epoch 33/100, Loss: 2.6865833774209023
Epoch 34/100, Loss: 2.7991332188248634
Epoch 35/100, Loss: 2.4492948576807976
Epoch 36/100, Loss: 2.557679757475853
Epoch 37/100, Loss: 2.8704196959733963
Epoch 38/100, Loss: 2.574683979153633
Epoch 39/100, Loss: 2.8233013451099396
Epoch 40/100, Loss: 2.6877714321017265
Epoch 41/100, Loss: 2.721966430544853
Epoch 42/100, Loss: 3.168629303574562
Epoch 43/100, Loss: 2.6732936203479767
Epoch 44/100, Loss: 2.5892612114548683


Epoch 45/100, Loss: 2.9001322463154793
Epoch 46/100, Loss: 2.6871967390179634
Epoch 47/100, Loss: 2.7428493201732635
Epoch 48/100, Loss: 2.9938071593642235
Epoch 49/100, Loss: 2.712011367082596
Epoch 50/100, Loss: 2.8108650743961334
Epoch 51/100, Loss: 2.860709309577942
Epoch 52/100, Loss: 2.7531924098730087
Epoch 53/100, Loss: 2.938736580312252
Epoch 54/100, Loss: 2.5878573954105377
Epoch 55/100, Loss: 2.9121720641851425
Epoch 56/100, Loss: 2.7917828112840652
Epoch 57/100, Loss: 2.9532700330018997
Epoch 58/100, Loss: 2.7115465328097343
Epoch 59/100, Loss: 2.735603727400303
Epoch 60/100, Loss: 2.6428380385041237


Epoch 61/100, Loss: 2.6677844747900963
Epoch 62/100, Loss: 2.680356189608574
Epoch 63/100, Loss: 2.6682445034384727
Epoch 64/100, Loss: 2.869970865547657
Epoch 65/100, Loss: 2.6211285293102264
Epoch 66/100, Loss: 3.43291637301445
Epoch 67/100, Loss: 2.684326581656933
Epoch 68/100, Loss: 2.843555100262165
Epoch 69/100, Loss: 2.6174879893660545
Epoch 70/100, Loss: 2.770047627389431
Epoch 71/100, Loss: 3.220469683408737
Epoch 72/100, Loss: 2.695634037256241
Epoch 73/100, Loss: 2.768889605998993
Epoch 74/100, Loss: 3.196025565266609
Epoch 75/100, Loss: 2.7289060950279236
Epoch 76/100, Loss: 2.8148601725697517
Epoch 77/100, Loss: 2.8472036868333817


...

Epoch 36/100, Loss: 2.3956732973456383
Epoch 37/100, Loss: 2.270946189761162
Epoch 38/100, Loss: 2.2097123190760612
Epoch 39/100, Loss: 2.2813028544187546
Epoch 40/100, Loss: 1.907286174595356
Epoch 41/100, Loss: 2.224848687648773
Epoch 42/100, Loss: 2.1647693812847137
Epoch 43/100, Loss: 2.098192445933819
Epoch 44/100, Loss: 2.092681907117367
Epoch 45/100, Loss: 2.3504093289375305
Epoch 46/100, Loss: 2.3210410103201866
Epoch 47/100, Loss: 2.3196315616369247
Epoch 48/100, Loss: 2.313885845243931
Epoch 49/100, Loss: 2.2917060777544975
Epoch 50/100, Loss: 2.3636240735650063
Epoch 51/100, Loss: 2.134551163762808


Epoch 52/100, Loss: 2.2966303527355194
Epoch 53/100, Loss: 2.2907037809491158
Epoch 54/100, Loss: 2.4770232886075974
Epoch 55/100, Loss: 2.0135822147130966
Epoch 56/100, Loss: 2.25774297863245
Epoch 57/100, Loss: 2.18903998285532
Epoch 58/100, Loss: 2.006791479885578
Epoch 59/100, Loss: 2.323603443801403
Epoch 60/100, Loss: 2.2508522048592567
Epoch 61/100, Loss: 2.7912189811468124
Epoch 62/100, Loss: 2.23425230383873
Epoch 63/100, Loss: 2.3027994632720947
Epoch 64/100, Loss: 2.1732853576540947
Epoch 65/100, Loss: 2.2497782856225967
Epoch 66/100, Loss: 2.182107597589493
Epoch 67/100, Loss: 2.032951757311821


Epoch 68/100, Loss: 2.1076555326581
Epoch 69/100, Loss: 2.5529400408267975
Epoch 70/100, Loss: 2.164357863366604
Epoch 71/100, Loss: 2.316819652915001
Epoch 72/100, Loss: 2.254817955195904
Epoch 73/100, Loss: 3.134787030518055
Epoch 74/100, Loss: 2.0960203781723976
Epoch 75/100, Loss: 1.9401936754584312
Epoch 76/100, Loss: 2.2766953259706497
Epoch 77/100, Loss: 2.303057722747326
Epoch 78/100, Loss: 2.116179123520851
Epoch 79/100, Loss: 2.2158252000808716
Epoch 80/100, Loss: 2.1441968455910683
Epoch 81/100, Loss: 1.9900124184787273
Epoch 82/100, Loss: 2.0303879603743553


Epoch 83/100, Loss: 2.1072545796632767
Epoch 84/100, Loss: 2.3276112228631973
Epoch 85/100, Loss: 2.1595479622483253
Epoch 86/100, Loss: 2.3704449236392975
Epoch 87/100, Loss: 2.370456337928772
Epoch 88/100, Loss: 2.32900919765234
Epoch 89/100, Loss: 2.230035714805126
Epoch 90/100, Loss: 2.430184133350849
Epoch 91/100, Loss: 2.185614585876465
Epoch 92/100, Loss: 2.3789619356393814
Epoch 93/100, Loss: 2.154071532189846
Epoch 94/100, Loss: 2.0817080438137054
Epoch 95/100, Loss: 2.349464423954487
Epoch 96/100, Loss: 2.362081900238991
Epoch 97/100, Loss: 2.1426259949803352


Epoch 98/100, Loss: 2.1706060990691185
Epoch 99/100, Loss: 2.210709735751152
Epoch 100/100, Loss: 2.2211694046854973
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.5295
